In [1]:
source(here::here('MG3.functions1.R'))
source(here::here('MG3.functions9.R'))


# Expression analysis

In [2]:
queryTxt='SELECT DISTINCT ensembl_gene_id FROM interestinggenes'
interestingGenes=dbGetQuery(mydbRO, queryTxt)[[1]]
queryTxt='SELECT DISTINCT ensembl_gene_id FROM AllSpeciesFoundGeneNames5MB'
allGenes=dbGetQuery(mydbRO, queryTxt)[[1]]

In [3]:
ordinaryGenes=setdiff(allGenes, interestingGenes)

In [4]:
protein_expr_DT=as.data.table(read.table(paste0(StaticInpDir, "protein_atlas/normal_ihc_data.tsv"), sep = '\t', header = TRUE))
#protein_expr_DT=protein_expr_DT[ protein_expr_DT$Gene %in% allGenes,] # keeping only the analyzed genes

Warning message in file(file, "rt"):
“cannot open file '/home/frimanav/data/lambda/data/protein_atlas/normal_ihc_data.tsv': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


In [ ]:
setdiff(interestingGenes, protein_expr_DT$Gene)

In [ ]:
setkey(protein_expr_DT, Gene)

In [ ]:
protein_expr_DT$Level.numb=NA

In [ ]:
#Ascending
#Descending
#High
#Low
#Medium
#N/A
#Not detected
#Not representative

protein_expr_DT$Level.numb[protein_expr_DT$Level == "Not detected"] =0 
protein_expr_DT$Level.numb[protein_expr_DT$Level == "Low"] =1
protein_expr_DT$Level.numb[protein_expr_DT$Level == "Medium"] =2
protein_expr_DT$Level.numb[protein_expr_DT$Level == "Ascending"] =2
protein_expr_DT$Level.numb[protein_expr_DT$Level == "Descending"] =2
protein_expr_DT$Level.numb[protein_expr_DT$Level == "High"] =3


In [ ]:
protein_expr_DT$interestingGene = protein_expr_DT$Gene %in% interestingGenes

In [ ]:
HMMid='ENSG00000001626'

JOBid=Sys.getenv("SLURM_JOB_ID")
tmpDir=paste0('/lscratch/',JOBid,"/")

queryTxt=sprintf('SELECT * FROM HMMstats5MB WHERE ensembl_gene_id="%s"', HMMid)
HMMstatsDT=as.data.frame(dbGetQuery(mydbRO, queryTxt))

In [ ]:
queryTxt='SELECT taxonomy.accession, taxonomy.id AS taxon FROM taxonomy INNER JOIN (SELECT accession, MAX(orderTax) AS maxOrd FROM taxonomy GROUP BY accession) AS maxrank 
        ON taxonomy.accession=maxrank.accession WHERE 
            taxonomy.orderTax=maxrank.maxOrd  AND 
            taxonomy.accession IN ( SELECT * FROM EutherianGenomes )'
taxID.DF=dbGetQuery(mydbRO, queryTxt)

In [ ]:
HMMstatsDT=merge(HMMstatsDT, taxID.DF, by="accession", all.x = TRUE, all.y = FALSE)

In [ ]:
# iterating over taxons and selecting the best assembly
OnePerTax=NULL
for (taxon in unique(HMMstatsDT$taxon)){
    boolmask=HMMstatsDT$taxon ==taxon
    tmpSubset=HMMstatsDT[ boolmask, ]
    OnePerTax=addToDF(OnePerTax, selectAssembly(tmpSubset))
}

In [ ]:
queryTxt=sprintf('SELECT DISTINCT * FROM taxonomy WHERE accession IN ("%s")',
                    paste0(OnePerTax$accession, collapse = '", "'))
PresentTaxizeDF=dbGetQuery(mydbRO, queryTxt)

In [ ]:
PresentTaxizeDF=merge(PresentTaxizeDF, OnePerTax[, c("accession", "taxon")], by="accession", all.x = TRUE)

In [ ]:
PresentTaxizeList=split(PresentTaxizeDF, PresentTaxizeDF$taxon)
PresentTaxizeList=lapply(PresentTaxizeList, FUN = function(x) {return(x[ order(x$orderTax),])})

In [ ]:
AlignedTaxaDF=BringTaxaToCommonCoordinateFrame( PresentTaxizeList )
AlignedTaxaDF.id=BringTaxaToCommonCoordinateFrame( PresentTaxizeList, indexesOnly = TRUE )


In [ ]:
currentLevel=ncol(AlignedTaxaDF)-1
prevTaxon=AlignedTaxaDF[1,ncol(AlignedTaxaDF)]

AlignedTaxaBranchingLevelsRaw=recursiveTreeBranchFinder(currentLevel, prevTaxon, AlignedTaxaDF, TRUE)
AlignedTaxaBranchingLevels=unique(sort(AlignedTaxaBranchingLevelsRaw))
AlignedTaxaLevelsToCheck=sort(unique(c(1, AlignedTaxaBranchingLevels)))


In [ ]:
AlignedTaxaDFSubset=AlignedTaxaDF[, AlignedTaxaLevelsToCheck]
AlignedTaxaDF.idSubset=AlignedTaxaDF.id[, AlignedTaxaLevelsToCheck]

In [ ]:
currentLevel=ncol(AlignedTaxaDFSubset)-1
prevTaxon=AlignedTaxaDFSubset[1,ncol(AlignedTaxaDFSubset)]
message(prevTaxon)

In [ ]:
returnTreeList=recursiveTreeBuilderV4(currentLevel, prevTaxon, AlignedTaxaDFSubset, AlignedTaxaDF.idSubset, OnePerTax, TRUE, parentID = 1)
returnTreeString=returnTreeList[["text"]]
returnTreeXML=returnTreeList[["data"]]

In [ ]:
edgeMarks=str_extract_all(string = returnTreeString, pattern = "\\{[^\\}]+@[0-9]+-[0-9]+\\}")
i=1
for (edgeMark  in unlist(edgeMarks)){
    returnTreeString=gsub(edgeMark, paste0("{",as.character(i),"}"), returnTreeString,fixed = TRUE)
    returnTreeXMLnew=gsub(edgeMark, as.character(i), returnTreeXML,fixed = TRUE)
    if (paste0(returnTreeXML, collapse = " ")==paste0(returnTreeXMLnew, collapse = " ")){
        message(edgeMark)
    }
    returnTreeXML=returnTreeXMLnew
    i=i+1
}
outputJSON=paste0('{\n"tree": "',returnTreeString,'",\n"data" : [')
for (XMLrecord in returnTreeXML){
    outputJSON=paste0(outputJSON,XMLrecord,"\n,")
}
outputJSON=substring(outputJSON,1,nchar(outputJSON)-1)
outputJSON=paste0(outputJSON,'],\n"metadata": {"info": "script by Alex", "data": "',getTimeStamp(),'"}}')
treeFileName=paste0(tmpDir,"tree",Sys.getpid(), ".json")
fileConn<-file(treeFileName)
writeLines(outputJSON, fileConn)
close(fileConn)

In [ ]:
options(repr.plot.width=40, repr.plot.height=45 )

jTREEtmp=read.jtree(file=treeFileName)
jTREEtmp@data$TaxonLevel<-as.integer(jTREEtmp@data$TaxonLevel)
jTREEtmp@data[jTREEtmp@data$node, ] = jTREEtmp@data
TEasNormDetectedTillNextCDS=as.numeric(jTREEtmp@data$asRMSKlen)/as.numeric(jTREEtmp@data$DSlen)
TEasNormDetectedTillNextCDS[abs(TEasNormDetectedTillNextCDS)==Inf]=0

isTip=(jTREEtmp@data$isTip==1)
MaskedNodes=jTREEtmp@data$node[jTREEtmp@data$isMasked==1] 


#jTREEtmp@phylo$tip.label=gsub(pattern = "_",replacement = " ",x = jTREEtmp@phylo$tip.label,fixed = TRUE)
#jTREEtmp@phylo$tip.label=paste(jTREEtmp@data$scientific.name[ jTREEtmp@data$node %in% 1:length(jTREEtmp@phylo$tip.label) ], "(",jTREEtmp@data$common.name[ jTREEtmp@data$node %in% 1:length(jTREEtmp@phylo$tip.label) ],")")

Nodelabels=rep("!!!!", max(jTREEtmp@data$node))
Nodelabels=jTREEtmp@data$text
Nodelabels[ 1:length(jTREEtmp@phylo$tip.label) ] = round(TEasNormDetectedTillNextCDS[isTip], 3)
Nodelabels[ MaskedNodes ] = NA


#adding the fixed (10K, 50K etc) dowstream length data
tmpTipData=jTREEtmp@data[ jTREEtmp@data$isTip==1,]
TipColnames<-colnames(tmpTipData)
tmpTipData$ID=jTREEtmp@phylo$tip.label
DSfixedLens=TipColnames[ grepl("asRMSKlen\\d+K", TipColnames)]
DSfixedLens=as.integer(str_remove_all(string = DSfixedLens, pattern = "[^0-9]"))
tipDownstreamTEasData=NULL
for (DSlen in DSfixedLens){
    asVal=as.numeric(tmpTipData[[sprintf("asRMSKlen%dK", DSlen)]])
    asFraction=asVal/(DSlen*1000)
    IDs=tmpTipData$ID
    tmpDF=data.frame(ID=IDs, asFraction=asFraction, DSlenFixed=DSlen)
    tipDownstreamTEasData=addToDF(tipDownstreamTEasData, tmpDF)
}


myTreeGG=ggtree::ggtree(jTREEtmp)+ geom_tiplab(as_ylab=TRUE, color='black')+
geom_label(aes(x=branch, label=Nodelabels, color=as.numeric(TEasNormDetectedTillNextCDS)))+ 
scale_color_gradient2(limits=c(0,max(as.numeric(na.exclude(as.numeric(TEasNormDetectedTillNextCDS))))),
                      oob = scales::squish,
                      low = "green", mid = "green", high = "red")

myTreeGG$data$DSlen[ MaskedNodes ] = NA
distanceToNextCDSLabels=round(as.numeric(myTreeGG$data$DSlen )/1000, digits = 1)
distanceToNextCDSLabels[ myTreeGG$data$isTip == 0] = NA

#adding taxonomy labels
curX=0
for (myColname in rev(colnames(AlignedTaxaDFSubset))){
    if (str_count(myColname, ".plus")>0) myColname="clade"
    myTreeGG=myTreeGG + annotate(geom = "text", x=curX-0.5, y=0, label=myColname)
    curX=curX+1
}

myTreeGG=myTreeGG+ theme(plot.margin = unit(c(1, 1.5, 1, 1), "inches")) + theme(legend.position = "bottom")+
labs(colour = "TE anti-sense")+
geom_fruit(data=tipDownstreamTEasData, geom=geom_tile,
                  mapping=aes(y=ID, x=factor(DSlenFixed), fill=asFraction),
                  offset = 0.04,size = 0.02, pwidth=0.1)+
scale_fill_gradient2(limits=c(0,max(as.numeric(na.exclude(as.numeric(TEasNormDetectedTillNextCDS))))),
                      oob = scales::squish,
                      low = "green", mid = "green", high = "red")+
new_scale_colour() +
geom_point(aes(x = myTreeGG$data$x, y = myTreeGG$data$y, 
               size=as.numeric(myTreeGG$data$DSlen ),
               color=log2(as.numeric(myTreeGG$data$asRMSKlen )/as.numeric(myTreeGG$data$sRMSKlen ))
              ))+scale_color_gradient2(
                      low = "white", mid = "gray", high = "black")+
labs(colour = "Strandedness, log2")+
annotate(geom = "text", x = myTreeGG$data$x + 0.25, 
         y = myTreeGG$data$y, label=distanceToNextCDSLabels) + 
annotate(geom = "text", x = 0, y = max(myTreeGG$data$y), label = HMMid, vjust = 2, hjust = -2)


myTreeGG

# Plotting a tree for every interesting gene

In [2]:
queryTxt='SELECT DISTINCT ensembl_gene_id FROM interestinggenes'
interestingGenes=dbGetQuery(mydbRO, queryTxt)[[1]]

In [3]:
# create dir
comToBash=paste0("mkdir -p ",OutputDir,"step9Trees/slurm_logs")
system(comToBash, intern = FALSE, wait = TRUE)
options(scipen = 999)
scriptsDir=normalizePath(path.expand(paste0(OutputDir,'../notebooks/')))
sbatchscript=sprintf("%s/Rscript.wrapper.lambdaenv.sh %s/MG3.Step9.PipelineForTreeGeneration.R", scriptsDir, scriptsDir)
swarmlines=paste(sbatchscript, interestingGenes)

swarmparams=paste0("#SWARM -t 1 -g 20 --time=01:00:00 --gres=lscratch:10 --partition=quick --logdir ", path.expand(OutputDir),"step9Trees/slurm_logs/")

swarmlines=c(swarmparams, swarmlines)
timestamp=getTimeStamp()
fname=paste0(OutputDir,"step9Trees/plot_interesting_genes-", timestamp,".swarm")
writeLines(text = swarmlines,  con=fname)
message("Run from bash: ", fname)

Run from bash: ~/data/lambda/output/step9Trees/plot_interesting_genes-2026-02-12_12.12.00.119328.swarm



In [4]:
swarmlines

[1] "#SWARM -t 1 -g 20 --time=01:00:00 --gres=lscratch:10 --partition=quick --logdir /home/frimanav/data/lambda/output/step9Trees/slurm_logs/"                  
 [2] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000001626"
 [3] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000029559"
 [4] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000081014"
 [5] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000091651"
 [6] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000094841"
 [7] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000111701"
 [8] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000111879"
 [9] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000112167"
[10] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000114767"
[11] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000115085"
[12] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000115364"
[13] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000115840"
[14] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000115993"
[15] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000116793"
[16] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000118690"
[17] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000121621"
[18] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000123901"
[19] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000124370"
[20] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000124678"
[21] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000130119"
[22] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000130385"
[23] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000133997"
[24] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.Step9.PipelineForTreeGeneration.R ENSG00000135324"
[25] "/vf/users/frimanav/lambda/notebooks/Rscript.wrapper.lambdaenv.sh /vf/users/frimanav/lambda/notebooks/MG3.St